## Schema & Type Fix Agent
•	Detects wrong data types (dates as text, numbers with commas, currency symbols). <br>
•	Standardizes formats (ISO dates, decimals, booleans). <br>
•	Flags columns with mixed types and proposes fixes.

Date formats

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_PATH = "../data/raw/amazon-purchases-sample.csv"

df = pd.read_csv(RAW_PATH)

print("Shape:", df.shape)
df.head()


Shape: (1820, 8)


,Order Date,Purchase Price Per Unit,Quantity,Shipping Address State,Title,ASIN/ISBN (Product Code),Category,Survey ResponseID
0,1/18/2018,17.16,1,PA,"NOW Foods, Certified Organic Better Stevia, Ex...",B005F9XFN0,SUGAR_SUBSTITUTE,GZ43D27
1,1/18/2018,FREE,1,PA,"F.M. Brown'S Encore Parakeet Food, 5-Pound",B000HHSHZG,PET_FOOD,RN79L07
2,1/18/2018,FREE,1,PA,COTTON CRAFT - Scandia Stripe 12 Pack - Pure C...,B014V1IYEM,TOWEL,KN18K36
3,1/18/2018,12.99,1,PA,"XREXS 4 Channels Digital Kitchen Timer Clock, ...",B01K4JKFK6,TIMER,JS10H76
4,1/18/2018,$12.87,1,pa,Premium Wrist Rests for Keyboard and Mouse Pad...,B01A92ETXM,WRIST_REST,EA67M16


In [2]:
df["Order Date"].dtype

dtype('O')

In [3]:
import re
import pandas as pd

def detect_date_format(date_str):
    if pd.isna(date_str):
        return "Missing"
    
    date_str = str(date_str).strip()
    
    patterns = {
        "YYYY-MM-DD": r"^\d{4}-\d{1,2}-\d{1,2}$",
        "MM/DD/YYYY": r"^\d{1,2}/\d{1,2}/\d{4}$",
        "MM-DD-YYYY": r"^\d{1,2}-\d{1,2}-\d{4}$",
        "DD-MMM-YYYY": r"^\d{1,2}-[A-Za-z]{3}-\d{4}$",
        "Month DD, YYYY": r"^[A-Za-z]+ \d{1,2}, \d{4}$"
    }
    
    for fmt, pattern in patterns.items():
        if re.match(pattern, date_str):
            return fmt
    
    return "Other/Unknown"


# Detect formats
df["Date_Format"] = df["Order Date"].apply(detect_date_format)

# Count occurrences
format_counts = df["Date_Format"].value_counts()

print("=== Different Date Formats Present ===")
print(format_counts)

# Show sample examples for each format
print("\n=== Sample Examples ===")

for fmt in format_counts.index:
    print(f"\nFormat: {fmt}")
    print(df[df["Date_Format"] == fmt]["Order Date"].head(3).tolist())


=== Different Date Formats Present ===
Date_Format
MM/DD/YYYY        1103
MM-DD-YYYY         665
Month DD, YYYY      33
Other/Unknown       19
Name: count, dtype: int64

=== Sample Examples ===

Format: MM/DD/YYYY
['1/18/2018', '1/18/2018', '1/18/2018']

Format: MM-DD-YYYY
['02-11-2018', '02-11-2018', '02-11-2018']

Format: Month DD, YYYY
['February 11, 2018', 'December 28, 2018', 'December 18, 2019']

Format: Other/Unknown
['28-Dec-18', '28-Jan-20', '03-Apr-20']


In [4]:
df["Detected_Date_Format"] = df["Order Date"].apply(detect_date_format)

format_counts = df["Detected_Date_Format"].value_counts()

format_counts

Detected_Date_Format
MM/DD/YYYY        1103
MM-DD-YYYY         665
Month DD, YYYY      33
Other/Unknown       19
Name: count, dtype: int64

Standardizes formats (ISO dates, decimals, booleans).

In [5]:
def detect_price_format(value):
    if pd.isna(value):
        return "Missing"
    
    value = str(value).strip()
    
    if re.match(r"^\d+(\.\d+)?$", value):
        return "Pure Numeric"
    
    if re.match(r"^\$\d+(\.\d+)?$", value):
        return "Dollar Format"
    
    if re.match(r"^₹\d+(\.\d+)?$", value):
        return "Rupee Format"
    
    if re.match(r"^\d{1,3}(,\d{3})*(\.\d+)?$", value):
        return "Comma Separated"
    
    if re.match(r"^\s+.*\s+$", str(value)):
        return "Whitespace Padded"
    
    if value.upper() in ["FREE", "N/A", "NA", "NONE"]:
        return "Text Value"
    
    return "Other/Unknown"


df["Price_Format"] = df["Purchase Price Per Unit"].apply(detect_price_format)

format_counts = df["Price_Format"].value_counts()

print("=== Different Price Formats Present ===")
print(format_counts)

print("\n=== Sample Examples Per Format ===")

for fmt in format_counts.index:
    print(f"\nFormat: {fmt}")
    examples = df[df["Price_Format"] == fmt]["Purchase Price Per Unit"].head(5).tolist()
    print(examples)



=== Different Price Formats Present ===
Price_Format
Pure Numeric     1593
Text Value         88
Dollar Format      71
Rupee Format       60
Other/Unknown       8
Name: count, dtype: int64

=== Sample Examples Per Format ===

Format: Pure Numeric
['17.16', '12.99', '4.33', '8.38', '4.49']

Format: Text Value
['FREE', 'FREE', 'FREE', 'FREE', 'FREE']

Format: Dollar Format
['$12.87', '$5.37', '$19.36', '$8.99', '$13.25']

Format: Rupee Format
['₹5.47', '₹4.99', '₹14.99', '₹13.99', '₹5.99']

Format: Other/Unknown
['₹FREE', '$$11.99', '$$3.59', '₹  12.49  ', '₹FREE']


## 2) Missing Value Treatment Agent
•	Profiles missingness (by column, segment, time). <br>
•	Suggests treatment rules (drop, impute, “unknown” bucket). <br>
•	Applies imputations with audit notes (what, why, % impacted).

In [6]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(RAW_PATH)

# Clean column names (avoid hidden space issues)
df.columns = df.columns.str.strip()

# Treat empty strings as missing
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# Calculate missing values
missing_summary = pd.DataFrame({
    "column": df.columns,
    "total_rows": len(df),
    "missing_count": df.isna().sum().values,
    "missing_percentage": (df.isna().mean() * 100).round(2).values
}).sort_values("missing_percentage", ascending=False)

print(missing_summary)


                     column  total_rows  missing_count  missing_percentage
6                  Category        1820             67                3.68
4                     Title        1820             67                3.68
3    Shipping Address State        1820             61                3.35
0                Order Date        1820              0                0.00
2                  Quantity        1820              0                0.00
1   Purchase Price Per Unit        1820              0                0.00
5  ASIN/ISBN (Product Code)        1820              0                0.00
7         Survey ResponseID        1820              0                0.00


In [7]:
print("Unique Survey ResponseID:", df["Survey ResponseID"].nunique())
print("Duplicate rows (full row):", df.duplicated().sum())
print("Duplicate Survey IDs:", df.duplicated(subset=["Survey ResponseID"]).sum())


Unique Survey ResponseID: 1816
Duplicate rows (full row): 4
Duplicate Survey IDs: 4


In [8]:
print("Exact duplicate rows (all columns):", df.duplicated().sum())

Exact duplicate rows (all columns): 4
